# Figure 2: Multi-Dataset Core Benchmark

- Figure / panels: `Fig2a`, `Fig2b`, `Fig2c`, `Fig2d`, `Fig2e`
- Inputs: `artifacts/results/<dataset>/metrics_*.csv`, `artifacts/results/<baseline>/<dataset>/metrics.csv`, payload `pkl`, raw dataset `h5ad`
- Outputs: `artifacts/paper_figures/main/Fig2_MultiDatasetBenchmark/*`
- Run order: run after the core benchmark results for the five single-perturbation datasets are ready
- Dataset selector: edit `SELECTED_DATASET_KEYS` in the parameter cell below
- Case selectors: edit `CASE_SHARED_BASELINE` and `CASE_GEARS_MISSING` in the parameter cell below
- Role: Main text
- Baseline note: `scGPT` is treated as a foundation-model baseline rather than a task-specific perturbation model


In [ ]:
from __future__ import annotations

import importlib
import sys
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis as paper_style_axis
from scripts.trishift.analysis import baseline_panel

apply_gears_paper_style(font_scale=1.0)
from scripts.trishift.analysis._result_adapter import load_payload_item
from trishift._utils import load_adata, load_yaml, normalize_condition
importlib.reload(baseline_panel)


In [ ]:
ALL_DATASET_SPECS = [
    {'dataset_key': 'dixit', 'dataset_label': 'Dixit'},
    {'dataset_key': 'adamson', 'dataset_label': 'Adamson'},
    {'dataset_key': 'norman', 'dataset_label': 'Norman (single)'},
    {'dataset_key': 'replogle_k562_essential', 'dataset_label': 'Replogle K562'},
    {'dataset_key': 'replogle_rpe1_essential', 'dataset_label': 'Replogle RPE1'},
]
AVAILABLE_DATASET_KEYS = [spec['dataset_key'] for spec in ALL_DATASET_SPECS]
SELECTED_DATASET_KEYS = ['adamson', 'norman']  # edit here
invalid_dataset_keys = [key for key in SELECTED_DATASET_KEYS if key not in AVAILABLE_DATASET_KEYS]
if invalid_dataset_keys:
    raise ValueError(f'Unknown dataset keys: {invalid_dataset_keys}. Available: {AVAILABLE_DATASET_KEYS}')
DATASET_SPECS = [spec for spec in ALL_DATASET_SPECS if spec['dataset_key'] in SELECTED_DATASET_KEYS]
if not DATASET_SPECS:
    raise ValueError('SELECTED_DATASET_KEYS produced an empty dataset list.')

MAIN_MODELS = ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt']  # edit here; scgpt is the foundation-model baseline
SIGNAL_MODELS = ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt', 'systema_nonctl_mean', 'systema_matching_mean']  # edit here; scgpt is the foundation-model baseline
AVAILABLE_SPLIT_IDS = [1, 2, 3, 4, 5]
SELECTED_SPLIT_IDS = AVAILABLE_SPLIT_IDS.copy()  # edit here
invalid_split_ids = [split_id for split_id in SELECTED_SPLIT_IDS if split_id not in AVAILABLE_SPLIT_IDS]
if invalid_split_ids:
    raise ValueError(f'Unknown split ids: {invalid_split_ids}. Available: {AVAILABLE_SPLIT_IDS}')
SPLIT_IDS = [int(split_id) for split_id in SELECTED_SPLIT_IDS]
PATHS_CFG = load_yaml(str(repo_root / 'configs' / 'paths.yaml'))

CASE_SHARED_BASELINE = {
    'dataset': 'adamson',
    'condition': 'CREB1+ctrl',
    'display_models': ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt'],
    'check_models': ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt'],
    'top_k': 12,
    'split_policy': 'first_available',
}
CASE_GEARS_MISSING = {
    'dataset': 'adamson',
    'condition': 'TIMM44+ctrl',
    'display_models': ['trishift_nearest', 'biolord', 'genepert', 'scgpt'],
    'check_models': ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt'],
    'top_k': 12,
    'split_policy': 'first_available',
    'unavailable_note': 'GEARS unavailable for this perturbation',
}
MODEL_LABELS = {
    'trishift_nearest': 'TriShift nearest',
    'biolord': 'biolord',
    'gears': 'GEARS',
    'genepert': 'GenePert',
    'scgpt': 'scGPT',
    'systema_nonctl_mean': 'Systema nonctl-mean',
    'systema_matching_mean': 'Systema matching-mean',
    'Truth': 'Truth',
}
MODEL_COLORS = {
    'TriShift nearest': '#4C78A8',
    'biolord': '#0073C2',
    'GEARS': '#D04E59',
    'GenePert': '#54A24B',
    'scGPT': '#B279A2',
    'Systema nonctl-mean': '#9A8C98',
    'Systema matching-mean': '#7A6FF0',
    'Truth': '#7F7F7F',
}
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'main' / 'Fig2_MultiDatasetBenchmark'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Selected datasets:', SELECTED_DATASET_KEYS)
print('Selected splits:', SPLIT_IDS)
print('Shared-baseline case:', CASE_SHARED_BASELINE)
print('GEARS-missing case:', CASE_GEARS_MISSING)


In [ ]:
def safe_run_baseline(dataset_key: str, models: list[str], tag: str) -> dict[str, object]:
    out_dir = OUT_ROOT / f'{tag}_{dataset_key}'
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        result = baseline_panel.run_baseline_panel(dataset=dataset_key, models=models, split_ids=SPLIT_IDS, out_root=out_dir)
        return {'status': 'ready', 'result': result, 'out_dir': out_dir}
    except Exception as exc:
        return {'status': 'pending', 'error': str(exc), 'out_dir': out_dir}


def _style_axis(ax: plt.Axes, *, grid_axis: str = 'y') -> None:
    paper_style_axis(ax, grid_axis=grid_axis)
    ax.set_axisbelow(True)


def render_metric_bar(summary_df: pd.DataFrame, metric: str, metric_label: str, models: list[str], out_path: Path) -> None:
    work = summary_df[summary_df['model_name'].isin(models)].copy()
    fig, ax = plt.subplots(figsize=(9.8, 4.9), dpi=220)
    if work.empty:
        ax.text(0.5, 0.5, f'No rows for {metric}', ha='center', va='center')
        ax.axis('off')
    else:
        present_labels = [MODEL_LABELS[m] for m in models if MODEL_LABELS[m] in set(work['label'])]
        sns.barplot(data=work, x='dataset_label', y=f'mean_{metric}', hue='label', hue_order=present_labels, palette=MODEL_COLORS, ax=ax, saturation=0.95)
        ax.set_xlabel('')
        ax.set_ylabel(metric_label)
        ax.set_title('Cross-dataset benchmark')
        ax.tick_params(axis='x', rotation=18)
        ax.legend(title='', frameon=False, ncol=min(3, len(present_labels)), loc='upper center', bbox_to_anchor=(0.5, 1.20))
        for patch in ax.patches:
            patch.set_edgecolor('white')
            patch.set_linewidth(0.8)
        _style_axis(ax)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)


def render_heatmap(table: pd.DataFrame, out_path: Path, title: str) -> None:
    fig, ax = plt.subplots(figsize=(max(11.5, table.shape[1] * 0.95), max(4.8, table.shape[0] * 0.72)), dpi=220)
    if table.empty:
        ax.text(0.5, 0.5, 'No perturbation-specific summary available', ha='center', va='center')
        ax.axis('off')
    else:
        sns.heatmap(table, annot=True, fmt='.3f', cmap=sns.diverging_palette(15, 145, as_cmap=True), center=0.0, linewidths=0.6, linecolor='white', cbar_kws={'label': 'Score'}, annot_kws={'fontsize': 8}, ax=ax)
        ax.set_title(title)
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.tick_params(axis='x', rotation=32)
        for tick in ax.get_xticklabels():
            tick.set_ha('right')
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)


@lru_cache(maxsize=None)
def load_dataset_context(dataset_key: str):
    adata = load_adata(PATHS_CFG['datasets'][dataset_key])
    cond_values = adata.obs['condition'].astype(str).map(normalize_condition).to_numpy()
    if 'gene_name' in adata.var.columns:
        gene_names = adata.var['gene_name'].astype(str).tolist()
    else:
        gene_names = adata.var_names.astype(str).tolist()
    gene_index = {gene_name: idx for idx, gene_name in enumerate(gene_names)}
    return adata, cond_values, gene_index


@lru_cache(maxsize=None)
def load_normalized_payload(dataset_key: str, model_name: str, split_id: int):
    _, payload = load_payload_item(dataset=dataset_key, model_name=model_name, split_id=int(split_id), condition=None)
    return {normalize_condition(str(k)): v for k, v in payload.items() if isinstance(v, dict)}


def resolve_case_item(dataset_key: str, condition_key: str, model_name: str, split_policy: str = 'first_available') -> dict[str, object]:
    if split_policy != 'first_available':
        raise ValueError(f'Unsupported split_policy: {split_policy}')
    errors: list[str] = []
    for split_id in SPLIT_IDS:
        try:
            payload = load_normalized_payload(dataset_key, model_name, int(split_id))
        except Exception as exc:
            errors.append(f'split {split_id}: {exc}')
            continue
        item = payload.get(condition_key)
        if item is not None:
            return {'model_name': model_name, 'status': 'ok', 'split_id': int(split_id), 'item': item, 'error': ''}
        errors.append(f'split {split_id}: condition missing')
    return {'model_name': model_name, 'status': 'missing', 'split_id': None, 'item': None, 'error': ' | '.join(errors[:4])}


def mean_vector(x) -> np.ndarray:
    return np.asarray(np.asarray(x, dtype=float).mean(axis=0)).reshape(-1)


def build_truth_reference(dataset_key: str, condition_key: str, reference_genes: list[str]) -> tuple[pd.DataFrame, np.ndarray]:
    adata, cond_values, gene_index = load_dataset_context(dataset_key)
    selected_genes = [gene_name for gene_name in reference_genes if gene_name in gene_index]
    if not selected_genes:
        raise ValueError(f'No reference genes could be aligned to raw dataset genes for {dataset_key}')
    idx = np.asarray([gene_index[gene_name] for gene_name in selected_genes], dtype=int)
    ctrl_mask = cond_values == 'ctrl'
    target_mask = cond_values == condition_key
    if int(target_mask.sum()) == 0:
        raise ValueError(f'Condition not found in raw data: {condition_key}')
    ctrl_mean = np.asarray(adata.X[ctrl_mask][:, idx].mean(axis=0)).reshape(-1).astype(float, copy=False)
    target_mean = np.asarray(adata.X[target_mask][:, idx].mean(axis=0)).reshape(-1).astype(float, copy=False)
    truth_delta = target_mean - ctrl_mean
    truth_df = pd.DataFrame({'Gene': selected_genes, 'Truth': truth_delta})
    return truth_df, ctrl_mean


def extract_prediction_series(item: dict[str, object]) -> pd.Series:
    if item.get('Pred_full') is not None and item.get('gene_name_full') is not None:
        pred_mean = mean_vector(item['Pred_full'])
        pred_genes = np.asarray(item['gene_name_full']).astype(str)
    else:
        pred_mean = mean_vector(item['Pred'])
        pred_genes = np.asarray(item.get('DE_name', [f'g{i}' for i in range(pred_mean.shape[0])])).astype(str)
    return pd.Series(pred_mean, index=pred_genes).groupby(level=0).mean()


def safe_corr(a: np.ndarray, b: np.ndarray) -> float:
    x = np.asarray(a, dtype=float).reshape(-1)
    y = np.asarray(b, dtype=float).reshape(-1)
    if x.size == 0 or y.size == 0:
        return float('nan')
    if np.allclose(np.std(x), 0.0) or np.allclose(np.std(y), 0.0):
        return float('nan')
    return float(np.corrcoef(x, y)[0, 1])


def build_case_bundle(case_cfg: dict[str, object]) -> tuple[dict[str, object], pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    dataset_key = str(case_cfg['dataset']).strip()
    condition_key = normalize_condition(str(case_cfg['condition']).strip())
    display_models = list(case_cfg['display_models'])
    check_models = list(case_cfg.get('check_models', display_models))
    split_policy = str(case_cfg.get('split_policy', 'first_available')).strip()
    top_k = int(case_cfg.get('top_k', 12))
    unavailable_note = str(case_cfg.get('unavailable_note', '')).strip()

    resolved = [resolve_case_item(dataset_key, condition_key, model_name, split_policy=split_policy) for model_name in check_models]
    resolved_df = pd.DataFrame([
        {
            'dataset': dataset_key,
            'condition': condition_key,
            'model_name': row['model_name'],
            'label': MODEL_LABELS.get(row['model_name'], row['model_name']),
            'status': row['status'],
            'split_id': row['split_id'],
            'error': row['error'],
        }
        for row in resolved
    ])
    resolved_ok = [row for row in resolved if row['status'] == 'ok' and row['model_name'] in display_models]
    if len(resolved_ok) < len(display_models):
        missing = {row['model_name'] for row in resolved if row['status'] != 'ok'}
        raise ValueError(f'Missing payload for case {condition_key}: {sorted(missing)}')

    reference_item = resolved_ok[0]['item']
    reference_series = extract_prediction_series(reference_item)
    truth_df, _ = build_truth_reference(dataset_key, condition_key, reference_series.index.astype(str).tolist())
    truth_series = truth_df.set_index('Gene')['Truth']

    ranking_parts = []
    metrics_rows = []
    plot_series = {'Truth': truth_series}
    for row in resolved_ok:
        label = MODEL_LABELS.get(row['model_name'], row['model_name'])
        pred_series = extract_prediction_series(row['item'])
        common_genes = truth_series.index.intersection(pred_series.index)
        aligned_truth = truth_series.loc[common_genes].astype(float)
        aligned_pred = pred_series.loc[common_genes].astype(float)
        diff = (aligned_pred - aligned_truth).abs()
        ranking_parts.append(diff.rename(label))
        plot_series[label] = pred_series
        metrics_rows.append({
            'dataset': dataset_key,
            'condition': condition_key,
            'model_name': row['model_name'],
            'label': label,
            'split_id': row['split_id'],
            'pearson_to_truth': safe_corr(aligned_truth.to_numpy(), aligned_pred.to_numpy()),
            'mae_to_truth': float(diff.mean()) if len(diff) else float('nan'),
        })

    ranking_df = pd.concat(ranking_parts, axis=1).fillna(0.0)
    top_genes = ranking_df.mean(axis=1).sort_values().index[:top_k]
    ordered_genes = truth_series.loc[top_genes].abs().sort_values(ascending=False).index.tolist()

    plot_rows = []
    for label, series in plot_series.items():
        aligned = series.reindex(ordered_genes).astype(float)
        plot_rows.append(pd.DataFrame({'Gene': ordered_genes, 'Expression': aligned.to_numpy(), 'Group': label}))
    plot_df = pd.concat(plot_rows, ignore_index=True)
    metrics_df = pd.DataFrame(metrics_rows).sort_values(['mae_to_truth', 'pearson_to_truth'], ascending=[True, False]).reset_index(drop=True)
    meta = {
        'dataset': dataset_key,
        'condition': condition_key,
        'top_k': top_k,
        'selected_splits': {row['model_name']: row['split_id'] for row in resolved},
        'unavailable_note': unavailable_note,
    }
    return meta, plot_df, metrics_df, resolved_df


def render_case_plot(plot_df: pd.DataFrame, title: str, out_path: Path, unavailable_note: str = '') -> None:
    plt.figure(figsize=(13.4, 5.2), dpi=220)
    ax = sns.barplot(data=plot_df, x='Gene', y='Expression', hue='Group', palette=MODEL_COLORS, errorbar=None, saturation=0.95)
    for patch in ax.patches:
        patch.set_edgecolor('white')
        patch.set_linewidth(0.7)
    ax.set_xlabel('')
    ax.set_ylabel('Expression change over control')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=32)
    ax.legend(title='', frameon=False, ncol=min(3, plot_df['Group'].nunique()), loc='upper center', bbox_to_anchor=(0.5, 1.18))
    ax.axhline(y=0, color='#4A4A4A', linewidth=0.8, linestyle='-')
    _style_axis(ax)
    if unavailable_note:
        ax.text(
            0.99,
            0.98,
            unavailable_note,
            transform=ax.transAxes,
            ha='right',
            va='top',
            fontsize=9,
            bbox={'boxstyle': 'round,pad=0.25', 'facecolor': 'white', 'edgecolor': '#666666', 'alpha': 0.9},
        )
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches='tight')
    plt.close()


In [ ]:
runs = []
for spec in DATASET_SPECS:
    benchmark_run = safe_run_baseline(spec['dataset_key'], MAIN_MODELS, 'benchmark')
    signal_run = safe_run_baseline(spec['dataset_key'], SIGNAL_MODELS, 'signal')
    runs.append({**spec, 'benchmark_status': benchmark_run['status'], 'benchmark_run': benchmark_run, 'signal_status': signal_run['status'], 'signal_run': signal_run})

availability_df = pd.DataFrame([{
    'dataset_key': row['dataset_key'],
    'dataset_label': row['dataset_label'],
    'benchmark_status': row['benchmark_status'],
    'signal_status': row['signal_status'],
    'benchmark_error': row['benchmark_run'].get('error', ''),
    'signal_error': row['signal_run'].get('error', ''),
} for row in runs])
availability_df.to_csv(OUT_ROOT / 'dataset_availability.csv', index=False, encoding='utf-8-sig')
display(availability_df)

benchmark_frames, signal_frames = [], []
for row in runs:
    if row['benchmark_status'] == 'ready':
        df = row['benchmark_run']['result']['summary_df'].copy(); df['dataset_label'] = row['dataset_label']; benchmark_frames.append(df)
    if row['signal_status'] == 'ready':
        df = row['signal_run']['result']['summary_df'].copy(); df['dataset_label'] = row['dataset_label']; signal_frames.append(df)
benchmark_summary_df = pd.concat(benchmark_frames, ignore_index=True) if benchmark_frames else pd.DataFrame()
signal_summary_df = pd.concat(signal_frames, ignore_index=True) if signal_frames else pd.DataFrame()
benchmark_summary_df.to_csv(OUT_ROOT / 'benchmark_summary_all.csv', index=False, encoding='utf-8-sig')
signal_summary_df.to_csv(OUT_ROOT / 'signal_summary_all.csv', index=False, encoding='utf-8-sig')
display(benchmark_summary_df.head())
display(signal_summary_df.head())


In [ ]:
render_metric_bar(benchmark_summary_df, 'pearson', 'Pearson', MAIN_MODELS, OUT_ROOT / 'fig2a_main_benchmark.png')

signal_metrics = ['mean_systema_corr_20de_allpert', 'mean_systema_corr_deg_r2', 'mean_scpram_r2_degs_mean_mean', 'mean_scpram_wasserstein_degs_sum']
metric_name_map = {
    'systema_corr_20de_allpert': 'Systema corr
(top20 DE)',
    'systema_corr_deg_r2': 'Systema corr
(DE R2)',
    'scpram_r2_degs_mean_mean': 'scPRAM R2
(DE mean)',
    'scpram_wasserstein_degs_sum': 'scPRAM Wass.
(DE sum)',
}
heatmap_df = signal_summary_df[signal_summary_df['model_name'].isin(SIGNAL_MODELS)].copy()
if not heatmap_df.empty:
    keep_cols = ['label', 'dataset_label'] + [c for c in signal_metrics if c in heatmap_df.columns]
    heatmap_df = heatmap_df[keep_cols].melt(id_vars=['label', 'dataset_label'], var_name='metric', value_name='value')
    heatmap_df['metric'] = heatmap_df['metric'].str.replace('mean_', '', regex=False).map(metric_name_map).fillna(heatmap_df['metric'])
    heatmap_df['column'] = heatmap_df['dataset_label'] + '
' + heatmap_df['metric']
    heatmap_table = heatmap_df.pivot_table(index='label', columns='column', values='value', aggfunc='mean')
else:
    heatmap_table = pd.DataFrame()
render_heatmap(heatmap_table, OUT_ROOT / 'fig2b_signal_heatmap.png', 'Signal-aware recovery across datasets')

relative_rows = []
if not benchmark_summary_df.empty:
    for dataset_label, g in benchmark_summary_df.groupby('dataset_label'):
        tri = g[g['model_name'] == 'trishift_nearest']
        competitors = g[g['model_name'] != 'trishift_nearest']
        if tri.empty or competitors.empty:
            continue
        tri_pearson = float(tri['mean_pearson'].iloc[0]) if 'mean_pearson' in tri.columns else np.nan
        tri_nmse = float(tri['mean_nmse'].iloc[0]) if 'mean_nmse' in tri.columns else np.nan
        best_comp_pearson = competitors.sort_values('mean_pearson', ascending=False).iloc[0]
        best_comp_nmse = competitors.sort_values('mean_nmse', ascending=True).iloc[0]
        relative_rows.append({'dataset_label': dataset_label, 'pearson_gain_vs_best': tri_pearson - float(best_comp_pearson['mean_pearson']), 'nmse_gain_vs_best': float(best_comp_nmse['mean_nmse']) - tri_nmse})
relative_df = pd.DataFrame(relative_rows)
relative_df.to_csv(OUT_ROOT / 'relative_improvement_summary.csv', index=False, encoding='utf-8-sig')
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.3), dpi=220)
if relative_df.empty:
    for ax in axes:
        ax.text(0.5, 0.5, 'No cross-dataset summary available', ha='center', va='center'); ax.axis('off')
else:
    sns.barplot(data=relative_df, x='dataset_label', y='pearson_gain_vs_best', color='#0072B2', ax=axes[0])
    axes[0].set_title('TriShift gain over best baseline')
    axes[0].set_ylabel('Pearson gain')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=18)
    _style_axis(axes[0])
    sns.barplot(data=relative_df, x='dataset_label', y='nmse_gain_vs_best', color='#009E73', ax=axes[1])
    axes[1].set_title('TriShift gain over best baseline')
    axes[1].set_ylabel('NMSE gain')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=18)
    _style_axis(axes[1])
fig.tight_layout(); fig.savefig(OUT_ROOT / 'fig2c_relative_improvement.png', bbox_inches='tight'); plt.close(fig)
display(relative_df)


In [ ]:
case_specs = [
    ('fig2d', CASE_SHARED_BASELINE, 'Fig2d: Shared-baseline case'),
    ('fig2e', CASE_GEARS_MISSING, 'Fig2e: GEARS-unavailable case'),
]

for case_prefix, case_cfg, case_title in case_specs:
    try:
        case_meta, case_plot_df, case_metrics_df, case_resolved_df = build_case_bundle(case_cfg)
    except Exception as exc:
        print(f'{case_prefix} failed: {exc}')
        continue

    case_plot_df.to_csv(OUT_ROOT / f'{case_prefix}_case_expression_values.csv', index=False, encoding='utf-8-sig')
    case_metrics_df.to_csv(OUT_ROOT / f'{case_prefix}_case_expression_metrics.csv', index=False, encoding='utf-8-sig')
    case_resolved_df.to_csv(OUT_ROOT / f'{case_prefix}_case_resolution.csv', index=False, encoding='utf-8-sig')

    title = f"{case_title} | {case_meta['dataset_key']} | {case_meta['condition']}"
    render_case_plot(
        case_plot_df,
        title=title,
        out_path=OUT_ROOT / f'{case_prefix}_case_expression.png',
        unavailable_note=str(case_meta.get('unavailable_note', '')),
    )

    print(case_meta)
    display(case_metrics_df)
    display(case_resolved_df)
    display(case_plot_df.head(24))

print(OUT_ROOT)
